# Artists based on your Spotify listening

This notebook fetches **your top artists** from Spotify (based on your recent plays) and shows their details, top tracks, and albums.

**Step 1:** Run the cell below to open the Spotify authorization page in your browser.

In [ ]:
import webbrowser
from pathlib import Path
from dotenv import load_dotenv
import os

# Load credentials from .env (copy .env.example to .env and add your Client ID and Secret)
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIFY_REDIRECT_URI", "http://127.0.0.1:8080/callback")
scopes = "user-top-read user-library-read"

auth_url = (
    f"https://accounts.spotify.com/authorize"
    f"?client_id={client_id}"
    f"&response_type=code"
    f"&redirect_uri={redirect_uri}"
    f"&scope={scopes}"
)
print("Opening Spotify authorization in your browser...")
print("After you click Agree, copy the code from the URL (after ?code= and before &)")
webbrowser.open(auth_url)

**Step 2:** Paste the authorization code below, then run this cell to fetch your top artists and their details.

In [ ]:
import requests
import base64
from pathlib import Path
from dotenv import load_dotenv
import os

# Load credentials from .env (do not commit .env)
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIFY_REDIRECT_URI", "http://127.0.0.1:8080/callback")

# Paste the code from the URL after you authorized (the part after ?code= and before &)
auth_code = 'AQAXkLC9Uz00wPBND-8CxgrcmnfqPM_fsBdu1H5ikdAagZtu9vmUnowQWy9rda5UNa16-chPOtfdmjyDB3OplgyK9w7JCTDksGtbu-dt-eY6Gkvg_rcTQGX32Xrf2lIMSVWYw0H8ucCS6WiypBgd0Z6o4yAzHPmh-u2pEyhuXkoL9AYdwgUi1ZPmMBceVWK8KA5UeE11dOz40F_WpmKDLvM0OsM'

# Exchange code for token
auth_str = f"{client_id}:{client_secret}"
auth_base64 = base64.b64encode(auth_str.encode('utf-8')).decode('utf-8')
response = requests.post(
    'https://accounts.spotify.com/api/token',
    headers={'Authorization': f'Basic {auth_base64}', 'Content-Type': 'application/x-www-form-urlencoded'},
    data={'grant_type': 'authorization_code', 'code': auth_code, 'redirect_uri': redirect_uri}
)
tokens = response.json()
access_token = tokens['access_token']

# --- Get YOUR top artists from Spotify (based on your listening) ---
# time_range: short_term (4 weeks), medium_term (6 months), long_term (several years)
top_artists_resp = requests.get(
    'https://api.spotify.com/v1/me/top/artists',
    headers={'Authorization': f'Bearer {access_token}'},
    params={'time_range': 'medium_term', 'limit': 20}
)
top_artists_data = top_artists_resp.json()
top_artists = top_artists_data.get('items', [])

if not top_artists:
    print("No top artists found. Try long_term time_range or make sure you've been listening.")
else:
    print(f"Your top {len(top_artists)} artists (based on your Spotify plays):\n")

def get_artist_top_tracks(artist_id):
    r = requests.get(f"https://api.spotify.com/v1/artists/{artist_id}/top-tracks",
        headers={'Authorization': f'Bearer {access_token}'}, params={'market': 'US'})
    return r.json().get('tracks', [])

def get_artist_albums(artist_id, limit=3):
    r = requests.get(f"https://api.spotify.com/v1/artists/{artist_id}/albums",
        headers={'Authorization': f'Bearer {access_token}'},
        params={'market': 'US', 'limit': limit, 'include_groups': 'album,single'})
    return r.json().get('items', [])

for i, artist in enumerate(top_artists, 1):
    name = artist.get('name', '?')
    aid = artist.get('id')
    genres = artist.get('genres', [])
    followers = (artist.get('followers') or {}).get('total', 0)
    print(f"--- {i}. {name} ---")
    print(f"    Genres: {', '.join(genres[:5]) if genres else 'N/A'}")
    print(f"    Followers: {followers:,}")
    top_tracks = get_artist_top_tracks(aid)
    if top_tracks:
        print(f"    Top tracks: {', '.join(t.get('name', '?') for t in top_tracks[:5])}")
    albums = get_artist_albums(aid)
    if albums:
        print(f"    Recent albums: {', '.join(a.get('name', '?') for a in albums)}")
    print()